In [ ]:
from gemini_output import generate_response, dataframe_to_json
import re 
import os
from collections import OrderedDict
import pandas as pd
from adjusted_price_data import get_adjusted_price_of_all_companies


def gemini_momentum_analysis():
    def prepare_pivot(adjusted_pivot_adjusted_pivot_df, column_name="Ticker"):
        adjusted_pivot_adjusted_pivot_df = adjusted_pivot_adjusted_pivot_df.copy()
        adjusted_pivot_adjusted_pivot_df[["Open", "High", "Low", "Close", "Volume"]] = (
            adjusted_pivot_adjusted_pivot_df[["Open", "High", "Low", "Close", "Volume"]]
            .apply(pd.to_numeric, errors="coerce")
            .astype(float)
        )
        adjusted_pivot_adjusted_pivot_df["Date"] = pd.to_datetime(adjusted_pivot_adjusted_pivot_df["Date"], errors="coerce")
        return adjusted_pivot_adjusted_pivot_df.pivot_table(index="Date", columns=column_name, values="Close", aggfunc="first")


    adjusted_data = get_adjusted_price_of_all_companies()
    adjusted_pivot_adjusted_pivot_adjusted_pivot_df = prepare_pivot(adjusted_data, column_name="Ticker")
    df = adjusted_pivot_adjusted_pivot_adjusted_pivot_df


    # 1. Updated periods: Added Daily and stopped at 6-Months
    periods = [
        ("1-D", 1), ("2-D", 2), ("3-D", 3), ("4-D", 4),
        ("1-W", 5), ("2-W", 10), ("3-W", 15), 
        ("1-M", 21), ("2-M", 42), ("3-M", 63), ("6-M", 126)
    ]

    def get_short_term_summary(df):
        # Data Cleaning: Convert index to datetime and fill missing prices
        df.index = pd.to_datetime(df.index)
        df = df.sort_index().ffill()
        
        summary_data = {}
        
        for label, days in periods:
            # Check if we have enough historical data for the period
            if len(df) > days:
                # Formula: (Latest Price / Price N days ago) - 1
                # .iloc[-1] is today, .iloc[-(days + 1)] is 'days' ago
                returns = ((df.iloc[-1] / df.iloc[-(days + 1)]) - 1) * 100
                
                # Sort and pick top 10 performers
                top_10 = returns.sort_values(ascending=False).head(10)
                
                # Create the string format "Ticker | Return%"
                summary_data[label] = [f"{t} | {r:.2f}%" for t, r in top_10.items()]
            else:
                summary_data[label] = ["Data N/A"] * 10
                
        return pd.DataFrame(summary_data)

    summary_df = get_summary_table(df)
    return summary_df



    # prompt_text = f"""
    # 🌞 **नमस्ते, नेपालका लगानीकर्ताहरू!** 🇳🇵🚀

    # आजको बजारको प्रदर्शनमा गहिरो विश्लेषण गर्न र आजको व्यापारमा लुकेका रत्नहरू र सर्तक संकेतहरू पत्ता लगाउन तयार हुनुहोस्। इन्भेस्टमेन्टका लागि केही महत्वपूर्ण विचारहरू र चाखलाग्दा विश्लेषणहरू ल्याउने छौं, केही इमोजीको साथ! ✨

    # तपाईंको डेटा यस JSON ढाँचामा प्रस्तुत गरिएको छ:

    # {dataframe_to_json(summary_df)}

    # **📊 विश्लेषणका प्रमुख बुँदाहरू:**

    # 1. **गतिको विश्लेषण:**
    # - कुन स्टकहरू **छोटो समयको अवधिमा** (जस्तै 1-दिन, 2-दिन, 3-दिन) **सबैभन्दा तेज गतिमा** बढेका छन्?
    # - कुन स्टकहरू **साप्ताहिक र मासिक** अवधिमा **सम्भावित गतिको परिवर्तन** देखाउँदै छन्?
    # - **लामो अवधिको गति** (जस्तै 3 महिना, 6 महिना)का आधारमा कुन स्टकहरूले **मूल्य परिवर्तनको संकेत** गरेको छ?

    # 2. **🔍 असामान्य वा महत्त्वपूर्ण निरीक्षणहरू:**
    # - के कुनै स्टकले **वृद्धि भएको भोल्युम** सँगै **गतिको तीव्रता** देखाएको छ?
    # - कुन स्टकहरूको **प्रदर्शन** सुधार भइरहेको छ र कुन स्टकले **संभावित गिरावट**को संकेत दिएको छ?
    # - के कुनै स्टकहरूले **बुलिस डाइवर्जन्स** वा **बेयर ट्रेन्ड** देखा रहे छन् जसले भविष्यको प्रदर्शनलाई असर पुर्याउन सक्छ?

    # 3. **💡 भोल्युम र मूल्यका लागि विचारहरू:**
    # - कुन स्टकहरूको **भोल्युम** र **मूल्य परिवर्तन**को सहसंबन्ध उच्च छ?
    # - कुन स्टकहरूले **बढ्दो भोल्युम** र **नकरात्मक गति** देखाइरहेका छन्, जसले **रिभर्सल** वा **संचय** को संकेत दिन सक्छ?

    # 4. **🚨 संभावित जोखिमहरू:**
    # - के कुनै स्टकहरू **उच्च जोखिम** का साथ **गतिको तीव्रता** देखाउँदै छन् जसले **मूल्यको गिरावट** संकेत गर्न सक्छ?
    # - कुन स्टकहरूको **आरएसआई** ७० भन्दा माथि छ र कुन स्टकहरूको **आरएसआई ३० भन्दा तल** छन्?

    # 5. **🧠 लगानीकर्ताहरूका लागि सुझावहरू:**
    # - आजको बजार डेटा अनुसार, लगानीकर्ताहरूलाई कहाँ आफ्नो ध्यान केन्द्रित गर्नुपर्छ (जस्तै, बलियो गति भएका स्टकहरू, वा सुधारको लागि हेर्नुपर्ने स्टकहरू)?

    # ---

    # 🚀 प्रवृत्तिहरू हेर्दै, सूचित हुँदै, र बजारको चालसँगै लगानी गर्नुहोस्! 💸📉📈
    # """

    # # Generate the response using Gemini
    # gemini_output = generate_response(prompt_text)
    # cleaned_response = re.sub(r'\*\*(.*?)\*\*', r'\1', gemini_output)
    # return cleaned_response

IndentationError: expected an indented block after function definition on line 9 (2688246499.py, line 10)